<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/templates/23_cross_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

    def forward(self, x_q, x_kv):
        # Q from x_q, K/V from x_kv, no causal mask
        B, seq_q, _ = x_q.size()
        _, seq_k, _ = x_kv.size()

        Q = self.W_q(x_q).reshape(B, seq_q, self.num_heads, self.d_k).transpose(1, 2) # B x num_heads x seq_q x d_k
        K = self.W_k(x_kv).reshape(B, seq_k, self.num_heads, self.d_k).transpose(1, 2) # B x num_heads x seq_k x d_k
        V = self.W_v(x_kv).reshape(B, seq_k, self.num_heads, self.d_k).transpose(1, 2) # B x num_heads x seq_k x d_k

        attention = torch.matmul(torch.softmax(torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(self.d_k), dim=-1), V) # B x num_heads x seq_q x d_k
        concat = attention.transpose(1, 2).contiguous().view(B, seq_q, self.d_model)
        mh_cross_attention = self.W_o(concat)
        return mh_cross_attention

In [5]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [6]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.2ms)
  ✅ [2/4] Q and KV different lengths (1.6ms)
  ✅ [3/4] No causal mask — all KV affects all Q (53.1ms)
  ✅ [4/4] Gradient flow (31.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (88.4ms total)
  Progress saved. Run status() to see your dashboard.

